# HL perp backtest

Run a historical ATR stop/target backtest for the selected coin directions. This notebook fetches public candles and funding only; it never signs or submits orders. The reusable engine is `desk.hl.backtest.py`.

## Highlights

Overall result, best and worst signal, PnL split into price, funding and fees, the exit mix, the worst trade, and skipped entries. Filled in by section 5 once the backtest has run (Run All does both).

In [ ]:
from desk.notebook_highlights import HighlightsPanel

highlights = HighlightsPanel()

## 1. Set signals and assumptions

Edit the rows for the markets and directions to test. HIP-3 markets may use a `dex` namespace such as `xyz` with an unqualified `coin` such as `AAPL`. Fee rates are required because they vary by account and trading tier; enter the current maker/taker rates in basis points.

In [ ]:
from dataclasses import replace
import pandas as pd

signals = pd.DataFrame([
    {"coin": "BTC", "side": "LONG", "dex": ""},
    # {"coin": "AAPL", "side": "SHORT", "dex": "xyz"},
])

LOOKBACK_DAYS = 365
STARTING_EQUITY = 10_000.0
MAKER_FEE_BPS = 1.44  # 0.0144%; update if your fee tier changes.
TAKER_FEE_BPS = 4.32  # 0.0432%; update if your fee tier changes.
HORIZON_DAYS = 5
ATR_PERIOD = 14
STOP_ATR_MULTIPLE = 1.25
REWARD_RISK_RATIO = 1.0

## 2. Load the backtest engine and validate configuration

In [ ]:
from desk.hl.backtest import BacktestSettings, backtest_signal
from desk.hl.trade_analytics import fetch_daily_candles, fetch_funding_history

if MAKER_FEE_BPS is None or TAKER_FEE_BPS is None:
    raise ValueError("Set both maker and taker fee assumptions in basis points before backtesting.")

settings = BacktestSettings(
    starting_equity=STARTING_EQUITY,
    horizon_days=HORIZON_DAYS,
    atr_period=ATR_PERIOD,
    stop_atr_multiple=STOP_ATR_MULTIPLE,
    reward_risk_ratio=REWARD_RISK_RATIO,
    maker_fee_bps=MAKER_FEE_BPS,
    taker_fee_bps=TAKER_FEE_BPS,
)
settings.validate()
signals

## 3. Fetch historical inputs

The funding API is paginated, so longer backtests include older hourly funding observations instead of using only one capped response.

In [ ]:
market_data = {}
for row in signals.to_dict("records"):
    coin = str(row["coin"]).strip()
    dex = str(row.get("dex", "") or "").strip()
    if ":" in coin:
        prefix, coin = coin.split(":", 1)
        dex = dex or prefix
    market_data[(coin, dex)] = {
        "candles": fetch_daily_candles(coin, dex, LOOKBACK_DAYS),
        "funding": fetch_funding_history(coin, dex, LOOKBACK_DAYS),
    }
[(key, len(value["candles"]), len(value["funding"])) for key, value in market_data.items()]

## 4. Run each signal's historical trades

Each market is tested independently with one position at a time for that market. The results are not a combined multi-asset portfolio equity curve; overlapping positions across different signals are not netted against the 20% aggregate live-risk cap.

In [ ]:
trade_rows = []
skipped_entries = []
for row in signals.to_dict("records"):
    coin = str(row["coin"]).strip()
    dex = str(row.get("dex", "") or "").strip()
    if ":" in coin:
        prefix, coin = coin.split(":", 1)
        dex = dex or prefix
    market = market_data[(coin, dex)]
    trades, skipped = backtest_signal(
        market["candles"],
        market["funding"],
        str(row["side"]),
        replace(settings, max_leverage=3.0 if dex else 4.0),
    )
    trade_rows.extend({"coin": coin, "dex": dex, **trade} for trade in trades)
    skipped_entries.extend({"coin": coin, "reason": message} for message in skipped)

trades_df = pd.DataFrame(trade_rows)
if trades_df.empty:
    raise ValueError("No trades were generated. Check history length, funding data, and signal settings.")
trades_df.head()

## 5. Review results and skipped entries

When stop and target both touch the same daily candle, the engine assumes the stop filled first. Funding PnL uses entry price as the hourly notional-price proxy; fees are applied using the configured maker/taker assumptions.

In [ ]:
summary = trades_df.groupby(["coin", "dex", "side"], dropna=False).agg(
    trades=("net_pnl", "size"),
    win_rate=("net_pnl", lambda values: (values > 0).mean()),
    average_price_pnl=("price_pnl", "mean"),
    average_funding_pnl=("funding_pnl", "mean"),
    average_fees=("fees", "mean"),
    average_net_pnl=("net_pnl", "mean"),
    total_net_pnl=("net_pnl", "sum"),
).reset_index()
display(summary)
display(pd.DataFrame(skipped_entries).head(20))

In [ ]:
from desk.hl.backtest import backtest_highlights

highlights.show(*backtest_highlights(trades_df, skipped_entries, STARTING_EQUITY))

## 6. Export trade-level results (optional)

Change the path only if you want to persist these results locally. No wallet credentials are needed.

In [ ]:
OUTPUT_PATH = None  # Example: "data/hl_backtest_results.csv"
if OUTPUT_PATH:
    trades_df.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved {len(trades_df)} trades to {OUTPUT_PATH}")